# Loan Default EDA Dataset
### Exploratory Data Analysis for Loan Default Classification

This notebook performs a complete, project-ready EDA on **Loan_Default.csv**.

**Target column:** `Status`  
- `0` = Non-default / approved-performing case  
- `1` = Default / higher-risk case, as represented by the dataset label

The EDA covers:
1. Dataset structure
2. Missing values
3. Duplicate analysis
4. Target class distribution
5. Numerical and categorical feature analysis
6. Default-wise comparisons
7. Correlation analysis
8. Key EDA findings

> Note: The notebook preserves the raw dataset for inspection and creates a separate EDA-ready cleaned copy. For final model training, imputation should ideally be fit only on the training set to avoid data leakage.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

possible_paths = [
    "/content/Loan_Default.csv",
    "/mnt/data/Loan_Default.csv",
    "Loan_Default.csv"
]

csv_path = next((p for p in possible_paths if __import__("os").path.exists(p)), None)

if csv_path is None:
    raise FileNotFoundError("Loan_Default.csv was not found. Upload it or place it in the notebook folder.")

df_raw = pd.read_csv(csv_path)

print("Dataset loaded from:", csv_path)
print("Original shape:", df_raw.shape)
display(df_raw.head())


## 1. Dataset Overview
We first inspect the number of rows and columns, column names, data types, and a sample of the dataset.


In [ ]:
print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

print("\nData Types:")
display(pd.DataFrame({
    "Column": df_raw.columns,
    "Data Type": df_raw.dtypes.astype(str).values
}))

display(df_raw.head())
display(df_raw.tail())


## 2. Missing Value Analysis
Missing values are important in this dataset, especially for interest-rate, upfront-charge, debt-to-income, property-value, LTV and income-related fields.


In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values": df_raw.isnull().sum(),
    "Missing Percentage": (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
}).sort_values("Missing Values", ascending=False)

display(missing_summary)


In [ ]:
missing_nonzero = missing_summary[missing_summary["Missing Values"] > 0]

plt.figure(figsize=(11, 5))
plt.bar(missing_nonzero.index, missing_nonzero["Missing Values"])
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Missing Count")
plt.xticks(rotation=55, ha="right")
plt.tight_layout()
plt.show()


## 3. Duplicate Analysis
Duplicate rows can bias analysis and model evaluation, so they are checked before continuing.


In [ ]:
duplicate_count = df_raw.duplicated().sum()
duplicate_percentage = duplicate_count / len(df_raw) * 100

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")


## 4. Create an EDA-Ready Cleaned Dataset
For exploratory analysis:
- duplicate rows are removed,
- the identifier column `ID` is excluded from feature analysis,
- missing categorical values are represented as `Unknown`,
- missing numerical values are filled with the median.

This cleaned table is suitable for EDA and visualization. Final model preprocessing should be performed within the train/test pipeline.


In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True).copy()

if "ID" in df.columns:
    df = df.drop(columns=["ID"])

categorical_cols = df.select_dtypes(include="object").columns.tolist()
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

for col in numerical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Cleaned shape:", df.shape)
print("Remaining missing values:", int(df.isnull().sum().sum()))
display(df.head())


## 5. Target Class Distribution — `Status`
The target distribution tells us whether the loan-default classes are balanced or imbalanced.


In [ ]:
status_counts = df["Status"].value_counts().sort_index()
status_pct = (status_counts / len(df) * 100).round(2)

target_table = pd.DataFrame({
    "Count": status_counts,
    "Percentage": status_pct
})
target_table.index = ["Status 0", "Status 1"]

display(target_table)

plt.figure(figsize=(7, 5))
bars = plt.bar(["Status 0", "Status 1"], status_counts.values)

for bar, value in zip(bars, status_counts.values):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value,
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.title("Loan Default Class Distribution")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


### Observation
The dataset is moderately imbalanced. Therefore, model evaluation should not depend only on accuracy. Precision, recall, F1-score, ROC-AUC and the confusion matrix should also be considered.


## 6. Numerical Feature Summary
We summarize important continuous/numerical variables such as loan amount, income, credit score, interest rate, LTV and debt-to-income ratio.


In [ ]:
important_numeric = [
    "loan_amount",
    "rate_of_interest",
    "property_value",
    "income",
    "Credit_Score",
    "LTV",
    "dtir1",
    "term"
]

important_numeric = [c for c in important_numeric if c in df.columns]

display(df[important_numeric].describe().T.round(2))


## 7. Numerical Features by Default Status
This comparison helps identify whether default and non-default records show different average financial characteristics.


In [ ]:
grouped_means = df.groupby("Status")[important_numeric].mean().T.round(2)
grouped_means.columns = ["Status 0", "Status 1"]
display(grouped_means)


## 8. Loan Amount Distribution


In [ ]:
plt.figure(figsize=(9, 5))

for status in [0, 1]:
    values = df.loc[df["Status"] == status, "loan_amount"]
    plt.hist(values, bins=45, alpha=0.5, density=True, label=f"Status {status}")

plt.xlim(0, df["loan_amount"].quantile(0.99))
plt.title("Loan Amount Distribution by Status")
plt.xlabel("Loan Amount")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Credit Score Distribution
Credit score is a major credit-risk feature and is expected to be informative for default analysis.


In [ ]:
plt.figure(figsize=(9, 5))

for status in [0, 1]:
    values = df.loc[df["Status"] == status, "Credit_Score"]
    plt.hist(values, bins=35, alpha=0.5, density=True, label=f"Status {status}")

plt.title("Credit Score Distribution by Status")
plt.xlabel("Credit Score")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Income Distribution
Income can contain strong outliers, so the plot is restricted to the 99th percentile for readability.


In [ ]:
plt.figure(figsize=(9, 5))

for status in [0, 1]:
    values = df.loc[df["Status"] == status, "income"]
    plt.hist(values, bins=45, alpha=0.5, density=True, label=f"Status {status}")

plt.xlim(0, df["income"].quantile(0.99))
plt.title("Income Distribution by Status")
plt.xlabel("Income")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Loan-to-Value (LTV) Distribution
LTV represents the loan amount relative to property value and is useful in understanding credit exposure.


In [ ]:
plt.figure(figsize=(9, 5))

for status in [0, 1]:
    values = df.loc[df["Status"] == status, "LTV"]
    plt.hist(values, bins=45, alpha=0.5, density=True, label=f"Status {status}")

plt.xlim(0, df["LTV"].quantile(0.99))
plt.title("Loan-to-Value Ratio Distribution by Status")
plt.xlabel("LTV")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Boxplots for Important Risk Features
Boxplots provide a direct comparison of the median, spread and outliers for the two target classes.


In [ ]:
box_features = ["loan_amount", "Credit_Score", "LTV", "dtir1"]

for feature in box_features:
    if feature not in df.columns:
        continue

    data0 = df.loc[df["Status"] == 0, feature]
    data1 = df.loc[df["Status"] == 1, feature]

    plt.figure(figsize=(7, 5))
    plt.boxplot(
        [data0, data1],
        tick_labels=["Status 0", "Status 1"],
        showfliers=False
    )
    plt.title(f"{feature} by Loan Default Status")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()


## 13. Categorical Feature Analysis
We inspect important categorical fields to understand how loan characteristics differ across default classes.


In [ ]:
important_categorical = [
    "loan_type",
    "loan_purpose",
    "Credit_Worthiness",
    "business_or_commercial",
    "occupancy_type",
    "credit_type",
    "age",
    "Region"
]

important_categorical = [c for c in important_categorical if c in df.columns]

for feature in important_categorical:
    print("\n", "="*70)
    print("Feature:", feature)
    table = pd.crosstab(df[feature], df["Status"], normalize="index").round(3) * 100
    table.columns = ["Status 0 (%)", "Status 1 (%)"]
    display(table)


## 14. Default Rate by Age Group


In [ ]:
if "age" in df.columns:
    age_default = df.groupby("age")["Status"].agg(["count", "mean"]).sort_index()
    age_default["Default Rate (%)"] = (age_default["mean"] * 100).round(2)

    display(age_default[["count", "Default Rate (%)"]])

    plt.figure(figsize=(10, 5))
    plt.bar(age_default.index.astype(str), age_default["Default Rate (%)"])
    plt.title("Default Rate by Age Group")
    plt.xlabel("Age Group")
    plt.ylabel("Default Rate (%)")
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()


## 15. Default Rate by Loan Type


In [ ]:
if "loan_type" in df.columns:
    loan_type_default = df.groupby("loan_type")["Status"].agg(["count", "mean"])
    loan_type_default["Default Rate (%)"] = (loan_type_default["mean"] * 100).round(2)

    display(loan_type_default[["count", "Default Rate (%)"]])

    plt.figure(figsize=(8, 5))
    plt.bar(loan_type_default.index.astype(str), loan_type_default["Default Rate (%)"])
    plt.title("Default Rate by Loan Type")
    plt.xlabel("Loan Type")
    plt.ylabel("Default Rate (%)")
    plt.tight_layout()
    plt.show()


## 16. Correlation Analysis
The target `Status` is included in the numerical correlation matrix to see which numeric variables have stronger linear relationships with the target.


In [ ]:
corr_cols = [
    "loan_amount",
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "term",
    "property_value",
    "income",
    "Credit_Score",
    "LTV",
    "dtir1",
    "Status"
]

corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()

plt.figure(figsize=(11, 8))
image = plt.imshow(corr, vmin=-1, vmax=1)
plt.colorbar(image)

plt.xticks(range(len(corr_cols)), corr_cols, rotation=55, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

plt.title("Correlation Matrix of Loan Features")
plt.tight_layout()
plt.show()


## 17. Features Most Correlated with Status
Absolute correlation is used only as an exploratory indicator. A low linear correlation does not mean a feature is useless, because nonlinear models can still learn complex relationships.


In [ ]:
status_corr = corr["Status"].drop("Status").sort_values(key=abs, ascending=False)

display(pd.DataFrame({
    "Correlation with Status": status_corr.round(4),
    "Absolute Correlation": status_corr.abs().round(4)
}))


# Final EDA Summary

The Loan Default dataset contains a large mixture of categorical and numerical loan characteristics. The main EDA findings are:

1. The raw dataset contains **148,670 rows and 34 columns**.
2. There are **no exact duplicate rows** in the original file.
3. Several financial variables contain substantial missing values, especially:
   - `Upfront_charges`
   - `Interest_rate_spread`
   - `rate_of_interest`
   - `dtir1`
   - `property_value`
   - `LTV`
   - `income`
4. The target `Status` contains approximately:
   - **112,031 records with Status 0**
   - **36,639 records with Status 1**
   - about **75.36% vs 24.64%**
5. The dataset is therefore moderately imbalanced.
6. Important numerical risk variables include:
   - loan amount
   - interest rate
   - income
   - credit score
   - LTV
   - debt-to-income ratio
7. Important categorical variables include:
   - loan type
   - loan purpose
   - credit worthiness
   - age
   - credit type
   - occupancy type
   - region
8. Accuracy alone should not be used for model evaluation because of the class imbalance.

## Recommended PPT EDA Slides
**Slide 1 – Dataset Overview & Missing Values**  
Shape, columns, missing-value summary and cleaning steps.

**Slide 2 – Target Class Distribution**  
Status 0 vs Status 1 counts and percentages.

**Slide 3 – Financial Feature Analysis**  
Loan amount, income, credit score, LTV and DTI distributions/boxplots.

**Slide 4 – Categorical & Correlation Analysis**  
Default rate by age/loan type plus correlation heatmap and key findings.
